# Model 3: Introduction of iceberg costs

In [2]:
import numpy as np
from scipy.optimize import root
import pandas as pd

## Functions

### Production

In [3]:
def output(A, L):
    return A * L

### Prices

In [4]:
def price_factory_gate(w, A):
    return w / A

def price_ij(t, w, A):
    p_factory = price_factory_gate(w, A)
    return p_factory[:, np.newaxis] * (1 + t)

### Household

In [5]:
def utility(beta, Z):
    return np.sum(beta * np.log(Z), axis=0)

def budget_constraint(p, Z, I):
    return np.sum(p * Z, axis=0) - I

def income(w, L_bar):
    return w * L_bar

def demand_ij(beta, I, p):
    return (beta * I) / p

def cobb_douglas_consumption_composite(beta, Z):
    return np.prod((Z / beta) ** beta, axis=0)

def unit_expenditure_price_index(beta, p):
    return np.prod(p ** beta, axis=0)

### Total demand

In [6]:
def output_demanded(t, Z):
    return np.sum(Z * (1 + t), axis=1)

def transport_absorbed(t, Z):
    return t * Z

### Market clearing / residuals

In [7]:
def goods_market_clearing(t, Y, Z):
    return Y - output_demanded(t, Z)

def value_market_clearing(t, w, L_bar, beta, A):
    I = income(w, L_bar)
    delivered_prices = price_ij(t, w, A)
    Z = demand_ij(beta, I, delivered_prices)

    revenue = np.sum(delivered_prices * Z, axis=1)
    supply = w * L_bar
    
    return revenue - supply

## Modelling values

In [8]:
L_bar = np.array([100, 100])
A = np.array([1.0, 1.0])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
t = np.array([[0, 0.2], [0.2, 0]])

## Benchmark

In [9]:
def system_residuals(w2):
    w = np.array([1.0, w2[0]])
    value_clearing = value_market_clearing(t, w, L_bar, beta, A)
    return value_clearing[1]

### Value market clearing

In [10]:
sol = root(system_residuals, [2])
print("Solution for w_2:", sol.x[0])

Solution for w_2: 1.0


In [11]:
system_residuals(sol.x)

0.0

In [12]:
w_sol = np.array([1.0, sol.x[0]])
p_sol = price_ij(t, w_sol, A)
I_sol = income(w_sol, L_bar)
Z_sol = demand_ij(beta, I_sol, p_sol)
Y_sol = output(A, L_bar)
Q_sol = cobb_douglas_consumption_composite(beta, Z_sol)
P_sol = unit_expenditure_price_index(beta, p_sol)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Demands:", Z_sol)
print("Real consumption:", Q_sol)
print("Consumer price index:", P_sol)
print("Output:", Y_sol)

Prices: [[1.  1.2]
 [1.2 1. ]]
Wages: [1. 1.]
Demands: [[60.         33.33333333]
 [33.33333333 60.        ]]
Real consumption: [92.96671848 92.96671848]
Consumer price index: [1.07565376 1.07565376]
Output: [100. 100.]


In [13]:
goods_res = goods_market_clearing(t, Y_sol, Z_sol)
print("Goods market residuals:", goods_res)

Goods market residuals: [0. 0.]


### Part 3.3: Comparison of the effects of transport costs

In [14]:
t_vals = [0, 0.05, 0.1, 0.2, 0.4]
results = []

for t in t_vals:
    t_matrix = np.array([[0, t], [t, 0]])

    def temp_residuals(w2):
        w = np.array([1.0, w2[0]])
        value_clearing = value_market_clearing(t_matrix, w, L_bar, beta, A)
        return value_clearing[1]

    sol = root(temp_residuals, [2])
    w_sol = np.array([1.0, sol.x[0]])

    I_sol = income(w_sol, L_bar)
    p_sol = price_ij(t_matrix, w_sol, A)
    Z_sol = demand_ij(beta, I_sol, p_sol)
    Q_sol = cobb_douglas_consumption_composite(beta, Z_sol)

    results.append({
        't': t,
        'w1': w_sol[0],
        'w2': w_sol[1],
        'Z12': Z_sol[0, 1],
        'Z21': Z_sol[1, 0],
        'Q1': Q_sol[0],
        'Q2': Q_sol[1]
    })

In [15]:
results_df = pd.DataFrame(results)

In [17]:
from lets_plot import *
LetsPlot.setup_html()

p1 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='w1')) + labs(y = r'\(w_1\)', title = "Relative wage in node 1") + \
      theme_bw()
p2 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='w2')) + labs(y = r'\(w_2\)', title = "Relative wage in node 2") + \
      theme_bw()
p3 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Z12')) + labs(y = r'\(Z_{12}\)', title = "Demand for node 1's good in node 2") + \
      theme_bw()
p4 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Z21')) + labs(y = r'\(Z_{21}\)', title = "Demand for node 2's good in node 1") + \
      theme_bw()
p5 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Q1')) + labs(y = r'\(Q_1\)', title = "Real consumption in node 1") + \
      theme_bw()
p6 = ggplot(results_df, aes(x='t')) + \
      geom_line(aes(y='Q2')) + labs(y = r'\(Q_2\)', title = "Real consumption in node 2") + \
      theme_bw()

grid = gggrid([p1, p2, p3, p4, p5, p6], ncol=2)

grid

ggsave(grid, "fig/comparison_transport_costs_3.svg")

'/Users/timnehls/Library/CloudStorage/OneDrive-Persönlich/uni/umucourses/MMES/lab2/lets-plot-images/fig/comparison_transport_costs_3.svg'

## 4. Explain why nominal expenditure shares do not respond to transport costs.

Under Cobb-Douglas utility, households always spend a fixed share of their income $\beta_{ij}$ on goods from region $i$, so $p_{ij} Z_{ij} / I_j = \beta_{ij}$. When we replace the delivered price with $p_{ij} = p_i (1 + t_{ij})$, the demand equation becomes $Z_{ij} = \beta_{ij} I_j / [p_i (1 + t_{ij})]$. This shows that when transport costs $t_{ij}$ rise, the higher price is balanced out by an exact drop in the physical amount $Z_{ij}$ received. As a result, the total money spent on each origin stays completely unchanged.

## 5. Explain why this feature may be unrealistic when goods from different origins are close substitutes.


In reality, when shipping costs make imported goods more expensive, consumers switch their spending to cheaper local alternatives if the goods are good substitutes. Cobb-Douglas cannot capture this behavior because its elasticity of substitution is fixed at one ($\sigma = 1$), meaning price increases are matched by an identical drop in quantity, keeping total money spent constant. 

## Part VI: Comparison of models III–IV

In [57]:
L_bar = np.array([100, 100])
A = np.array([1.0, 1.0])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])
t = np.array([[0, 0.1], [0.1, 0]])

In [58]:
def system_residuals(w2):
    w = np.array([1.0, w2[0]])
    value_clearing = value_market_clearing(t, w, L_bar, beta, A)
    return value_clearing[1]

In [59]:
sol = root(system_residuals, [2])
print("Solution for w_2:", sol.x[0])

Solution for w_2: 1.0


In [ ]:
w_sol = np.array([1.0, sol.x[0]])
p_sol = price_ij(t, w_sol, A)
I_sol = income(w_sol, L_bar)
Z_sol = demand_ij(beta, I_sol, p_sol)
Y_sol = output(A, L_bar)
Q_sol = cobb_douglas_consumption_composite(beta, Z_sol)
P_sol = unit_expenditure_price_index(beta, p_sol)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Demands:", Z_sol)
print("Real consumption:", Q_sol)
print("Consumer price index:", P_sol)
print("Output:", Y_sol)

Prices: [[1.  1.1]
 [1.1 1. ]]
Wages: [1. 1.]
Demands: [[60.         36.36363636]
 [36.36363636 60.        ]]
Real consumption: [96.25935027 96.25935027]
Consumer price index: [1.03886012 1.03886012]
Output: [100. 100.]


In [61]:
goods_res = goods_market_clearing(t, Y_sol, Z_sol)
print("Goods market residuals:", goods_res)

Goods market residuals: [0. 0.]
